In [1]:
import numpy as np

# Network graph: 4 routers with base latency (ms) between connected links
# Nodes: 0 (Source), 1, 2, 3 (Destination)
routers = 4
latency = np.array([
    [0,  10, 50, inf := 999],  # Router 0 connections
    [10, 0,  15, 20],        # Router 1 connections
    [50, 15, 0,  10],        # Router 2 connections
    [inf, 20, 10, 0]         # Router 3 connections
])

# Initialize pheromones on all valid links
pheromone = np.where(latency < 999, 1.0, 0.0)
alpha, beta, rho = 1.0, 2.0, 0.1  # Pheromone importance, latency importance, decay rate

def send_ant_packet(src=0, dest=3):
    current, path, total_latency = src, [src], 0
    while current != dest:
        # Find neighboring routers
        neighbors = [i for i in range(routers) if latency[current][i] < 999 and i not in path]
        if not neighbors:
            return None, float('inf')  # Dead end

        # Compute router selection probabilities based on pheromones and inverse latency
        p = np.array([(pheromone[current][n]**alpha) * ((1.0 / latency[current][n])**beta) for n in neighbors])
        p /= p.sum()

        # Pick next hop
        nxt = np.random.choice(neighbors, p=p)
        total_latency += latency[current][nxt]
        path.append(nxt)
        current = nxt

    return path, total_latency

# --- Simulate Network Traffic ---
best_path, min_latency = None, float('inf')

for cycle in range(50):
    path, delay = send_ant_packet()
    
    if path and delay < min_latency:
        best_path, min_latency = path, delay

    # 1. Evaporate pheromones across network
    pheromone *= (1.0 - rho)

    # 2. Deposit pheromone based on path quality (lower latency = higher deposit)
    if path:
        deposit = 100.0 / delay
        for i in range(len(path) - 1):
            u, v = path[i], path[i+1]
            pheromone[u][v] += deposit
            pheromone[v][u] += deposit

print(f"Optimal Network Path: {' -> '.join(map(str, best_path))}")
print(f"Total Latency: {min_latency} ms")

Optimal Network Path: 0 -> 1 -> 3
Total Latency: 30 ms
